In [3]:
# ==========================================
# STEP 1: IMPORT LIBRARIES
# ==========================================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import ModelCheckpoint

# ==========================================
# STEP 2: LOAD DATASET
# ==========================================
df = pd.read_csv('Churn_Modelling.csv')

# Drop unnecessary columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode categorical columns
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])
df['Geography'] = LabelEncoder().fit_transform(df['Geography'])

# Features and target
X = df.drop('Exited', axis=1)
y = df['Exited']

# ==========================================
# STEP 3: TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================
# STEP 4: FEATURE SCALING
# ==========================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# STEP 5: BUILD MODEL FUNCTION
# ==========================================
def build_model(optimizer='adam'):
    model = Sequential()
    model.add(Dense(16, activation='relu', input_dim=X_train.shape[1]))
    model.add(Dense(8, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# ==========================================
# STEP 6: MANUAL GRID SEARCH (OPTIMIZERS)
# ==========================================
print("\n--- Manual Grid Search ---")

best_acc = 0
best_opt = ""

for opt in ['adam', 'rmsprop']:
    model = build_model(optimizer=opt)
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(opt, "Accuracy:", acc)

    if acc > best_acc:
        best_acc = acc
        best_opt = opt

print("\nBest Optimizer:", best_opt)

# ==========================================
# STEP 7: K-FOLD CROSS VALIDATION
# ==========================================
print("\n--- K-Fold Cross Validation ---")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = build_model(optimizer=best_opt)
    model.fit(X_tr, y_tr, epochs=10, batch_size=32, verbose=0)

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    cv_scores.append(acc)

print("Average K-Fold Accuracy:", np.mean(cv_scores))

# ==========================================
# STEP 8: MODEL CHECKPOINT
# ==========================================
print("\n--- Training Final Model with Checkpoint ---")

checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min'
)

final_model = build_model(optimizer=best_opt)

final_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[checkpoint],
    verbose=1
)

# ==========================================
# STEP 9: FINAL EVALUATION
# ==========================================
loss, accuracy = final_model.evaluate(X_test, y_test)
print("\nFinal Test Accuracy:", accuracy)


--- Manual Grid Search ---


C:\Users\MANAN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


adam Accuracy: 0.8565000295639038
rmsprop Accuracy: 0.8504999876022339

Best Optimizer: adam

--- K-Fold Cross Validation ---
Average K-Fold Accuracy: 0.8434999942779541

--- Training Final Model with Checkpoint ---
Epoch 1/20
245/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7461 - loss: 0.5707

250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7824 - loss: 0.5177 - val_accuracy: 0.8115 - val_loss: 0.4368
Epoch 2/20
237/250 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8047 - loss: 0.4478

250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8146 - loss: 0.4381 - val_accuracy: 0.8195 - val_loss: 0.4155
Epoch 3/20
238/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8253 - loss: 0.4212

...

250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8553 - loss: 0.3469 - val_accuracy: 0.8580 - val_loss: 0.3552
Epoch 14/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8564 - loss: 0.3452 - val_accuracy: 0.8580 - val_loss: 0.3576
Epoch 15/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8570 - loss: 0.3442 - val_accuracy: 0.8600 - val_loss: 0.3559
Epoch 16/20
235/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8602 - loss: 0.3445

250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8558 - loss: 0.3431 - val_accuracy: 0.8565 - val_loss: 0.3533
Epoch 17/20
244/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8666 - loss: 0.3285

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8565 - loss: 0.3418 - val_accuracy: 0.8605 - val_loss: 0.3527
Epoch 18/20
245/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8585 - loss: 0.3384

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8581 - loss: 0.3405 - val_accuracy: 0.8560 - val_loss: 0.3497
Epoch 19/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8569 - loss: 0.3396 - val_accuracy: 0.8585 - val_loss: 0.3502
Epoch 20/20
243/250 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8543 - loss: 0.3430

250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8581 - loss: 0.3381 - val_accuracy: 0.8565 - val_loss: 0.3483
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8565 - loss: 0.3483

Final Test Accuracy: 0.8565000295639038
